In [ ]:
import gc
gc.collect()

import torch
torch.cuda.empty_cache()

In [ ]:
import io
import os
import imageio
import cv2
import random
import collections
from collections import Counter
import ipywidgets
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
# 재현을 위한 시드 설정
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    tf.random.set_seed(seed)

seed_everything()

## Hyperparameters

In [ ]:
loaded_parameters = {}

with open('parameters.txt', 'r') as file:
    lines = file.readlines()
    for line in lines:
        key, value = line.strip().split(": ", 1)  # 한 번만 분할하도록 수정
        if value.startswith("{") and value.endswith("}"):
            value = eval(value)  # 딕셔너리 형태의 값은 eval() 사용
        else:
            try:
                value = eval(value)  # 다른 값들은 eval() 사용 시도
            except:
                value = value  # eval()로 변환할 수 없는 경우 문자열 그대로 사용
        loaded_parameters[key] = value

print(loaded_parameters)

In [ ]:
## 불러와지지 않을 경우 사용
# DATA
DATA_PATH = "action"         # 데이터 경로
NUM_CLASSES = 3                # 데이터셋 클래스 개수
LABELS = ['Clapping', 'Sipping', 'Stacking_Rings']
SPLIT_RATIOS = {"train": 0.7, "val": 0.15, "test": 0.15}
frame_interval = 5
min_frame_count = float('12')  # 각 비디오에서 가져올 최소 프레임 수 <- 구축
image_size = (60,60)

BATCH_SIZE = 16                 # BATCH_SIZE: 한 번의 forward/backward pass에서 처리되는 샘플의 수를 결정
AUTO = tf.data.AUTOTUNE         # tf.data.AUTOTUNE): TensorFlow가 자동으로 데이터 프리페칭 버퍼 크기를 조정하여 훈련 중 CPU와 GPU 활용도를 최적화하는 기능
INPUT_SHAPE = (12, 60, 60, 3)   # 3D image size는 28x28x28, 단일채널(grayscale) <- 학습

# OPTIMIZER
LEARNING_RATE = 1e-4            # train optimizer의 학습률 지정 : 0.0001
WEIGHT_DECAY = 1e-5             # 가중치 감쇠 지정(과적합을 방지하기 위한 정규화 기법) : 0.00001

# TRAINING
EPOCHS = 60                     # 반복학습 : 60

# TUBELET EMBEDDING
PATCH_SIZE = (4, 4, 4)                                # patch image size는 8x8x8
NUM_PATCHES = (INPUT_SHAPE[0] // PATCH_SIZE[0]) ** 2  # 패치 수 계산 : {(28x28x28)//(8x8x8) = 3}**2 = 9

# ViViT ARCHITECTURE
LAYER_NORM_EPS = 1e-6           # nomalizaion layer에서 사용되는 epsilson 값 지정 <- 0으로 나누는 것을 피하기 위함
PROJECTION_DIM = 128            # 각 패치는 128차원 벡터로 임베딩
NUM_HEADS = 8                   # multi head-attention에서 각 어텐션 헤드의 개수를 지정 : 8개
NUM_LAYERS = 8                  # transformer 레이어 개수 지정 : 8개

## Data Load

In [ ]:
train_data = np.load('inputdata/train_with_labels_second.npz')
train_videos = np.array(train_data['videos'])
train_labels = np.array(train_data['labels'])

In [ ]:
valid_data = np.load('inputdata/valid_with_labels_second.npz')
valid_videos = valid_data['videos']
valid_labels = valid_data['labels']

In [ ]:
test_data = np.load('inputdata/test_with_labels_second.npz')
test_videos = test_data['videos']
test_labels = test_data['labels']

## tf.data pipeline

In [ ]:

# 프레임 텐서와 레이블 전처리 함수 정의
@tf.function
def preprocess(frames: tf.Tensor, label: tf.Tensor):
    # images 픽셀값 정규화 : [0, 1] 범위의 부동소수점으로 변환
    frames = tf.image.convert_image_dtype(
        frames[
            ..., tf.newaxis
        ],  # newaxis: 기존차원을 유지하면서 새로운 차원 추가(Conv3D -> 4D로 확장)
        tf.float32,
    )
    # label 데이터타입 변환
    label = tf.cast(label, tf.float32)
    return frames, label


# 데이터로더 생성 함수 정의
def prepare_dataloader(
    videos: np.ndarray,
    labels: np.ndarray,
    loader_type: str = "train",
    batch_size: int = BATCH_SIZE,
):
    # 데이터셋 생성
    dataset = tf.data.Dataset.from_tensor_slices((videos, labels))

    # train인 경우 데이터 셔플
    if loader_type == "train":
        dataset = dataset.shuffle(BATCH_SIZE * 2)

    dataloader = (
        dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE) # prefetch :  데이터를 비동기적으로 미리 읽어오게 gka
    )
    return dataloader

In [ ]:
# 데이터로드 정의 함수 사용
trainloader = prepare_dataloader(train_videos, train_labels, "train")
validloader = prepare_dataloader(valid_videos, valid_labels, "valid")
testloader = prepare_dataloader(test_videos, test_labels, "test")

## Tubelet Embedding

In [ ]:
# 임베딩 레이어 정의 클래스
class TubeletEmbedding(layers.Layer):
    def __init__(self, embed_dim, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.projection = layers.Conv3D(
            filters=embed_dim,      # 차원 수 <- transformer input 차원과 동일해야 함
            kernel_size=patch_size, # 필터 크기는 패치 크기로
            strides=patch_size,     # 보폭도 패치크기로
            padding="VALID",        # 패딩 안함
        )
        self.flatten = layers.Reshape(target_shape=(-1, embed_dim))

    def call(self, videos):
        projected_patches = self.projection(videos) # 임베딩 수행
        flattened_patches = self.flatten(projected_patches) # 3D -> 2D 형태로 평면화
        return flattened_patches

## Positional Embedding

In [ ]:
class PositionalEncoder(layers.Layer):
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim

    # 레이어를 쌓기 위한 method
    def build(self, input_shape):
        _, num_tokens, _ = input_shape
        self.position_embedding = layers.Embedding(
            #input차원은 토큰개수로, output차원은 임베딩된 패치 차원 수로
            input_dim=num_tokens, output_dim=self.embed_dim
        )
        # 토큰 위치를 나타내는 positions 생성
        self.positions = tf.range(start=0, limit=num_tokens, delta=1)

    def call(self, encoded_tokens):
        # Encode the positions and add it to the encoded tokens
        encoded_positions = self.position_embedding(self.positions)
        encoded_tokens = encoded_tokens + encoded_positions
        return encoded_tokens

## Video Vision Transformer
- Spatio-temporal attention

In [ ]:
# 분류 모델 생성 함수 정의
def create_vivit_classifier(
    tubelet_embedder,
    positional_encoder,
    input_shape=INPUT_SHAPE,
    transformer_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    embed_dim=PROJECTION_DIM,
    layer_norm_eps=LAYER_NORM_EPS,
    num_classes=NUM_CLASSES,
):
    # 함수 동작 과정
    inputs = layers.Input(shape=input_shape)  # input
    patches = tubelet_embedder(inputs)          # 패치 임베딩
    encoded_patches = positional_encoder(patches) # 패치 위치정보 추가

    # Transformer block 반복 생성하며 시공간적 특성 학습
    for _ in range(transformer_layers):
        # 정규화 and MultiHeadAttention
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=0.1
        )(x1, x1)
        # Skip connection로 기존 값 보존
        x2 = layers.Add()([attention_output, encoded_patches])

        # 정규화 and MLP
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = keras.Sequential(
            [
                layers.Dense(units=embed_dim * 4, activation=tf.nn.gelu),
                layers.Dense(units=embed_dim, activation=tf.nn.gelu),
            ]
        )(x3)
        # Skip connection로 기존 값 보존
        encoded_patches = layers.Add()([x3, x2])

    # 레이어 정규화 and Global average pooling로 특성 요약
    representation = layers.LayerNormalization(epsilon=layer_norm_eps)(encoded_patches)
    representation = layers.GlobalAvgPool1D()(representation)

    # dense층에서 softmax 활성화함수 적용 후 분류 결과 출력
    outputs = layers.Dense(units=num_classes, activation="softmax")(representation)

    # 모델 생성 후 반환
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

## Train

In [ ]:
# 모델 훈련 함수 정의
def run_experiment():
    # model 초기화
    model = create_vivit_classifier(
        tubelet_embedder=TubeletEmbedding(
            embed_dim=PROJECTION_DIM, patch_size=PATCH_SIZE
        ),
        positional_encoder=PositionalEncoder(embed_dim=PROJECTION_DIM),
    )

    # 모델 컴파일
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    model.compile(
        # optimizer : Adam
        optimizer=optimizer,
        # loss function
        loss="sparse_categorical_crossentropy",
        # 평가지표
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            keras.metrics.SparseTopKCategoricalAccuracy(5, name="top-5-accuracy"),
        ],
    )

    # trainloader를 사용하여 모델 훈련
    _ = model.fit(trainloader, epochs=EPOCHS, validation_data=validloader)
    # testloader를 사용해 모델 평가 + 상위5개 예측 정확도 출력
    _, accuracy, top_5_accuracy = model.evaluate(testloader)
    print(f"Test accuracy: {round(accuracy * 100, 2)}%")
    print(f"Test top 5 accuracy: {round(top_5_accuracy * 100, 2)}%")

    # 모델 반환
    return model

In [ ]:
# 정의 함수를 실행해 훈련시킨 모델을 model에 저장
model = run_experiment()

In [ ]:
model.save('child_action_model_vivit.h5')

## Inference

In [ ]:
NUM_SAMPLES_VIZ = 10 # 시각화 샘플 개수
testsamples, labels = next(iter(testloader))
testsamples, labels = testsamples[:NUM_SAMPLES_VIZ], labels[:NUM_SAMPLES_VIZ]

ground_truths = []
preds = []
videos = []

for i, (testsample, label) in enumerate(zip(testsamples, labels)):
    # Generate gif
    with io.BytesIO() as gif:
        imageio.mimsave(gif, (testsample.numpy()).reshape((12, 60, 60, 3)).astype(np.uint8), "GIF", duration=100)
        videos.append(gif.getvalue())

    # Get model prediction
    output = model.predict(tf.expand_dims(testsample.numpy().reshape(12, 60, 60, 3), axis=0))[0]
    pred = np.argmax(output, axis=0)

    ground_truths.append(label.numpy().astype("int"))
    preds.append(pred)

def make_box_for_grid(image_widget, fit):
    # Make the caption
    if fit is not None:
        fit_str = "'{}'".format(fit)
    else:
        fit_str = str(fit)

    h = ipywidgets.HTML(value="" + str(fit_str) + "")

    # Make the green box with the image widget inside it
    boxb = ipywidgets.widgets.Box()
    boxb.children = [image_widget]

    # Compose into a vertical box
    vb = ipywidgets.widgets.VBox()
    vb.layout.align_items = "center"
    vb.children = [h, boxb]
    return vb


boxes = []
for i in range(NUM_SAMPLES_VIZ):
    ib = ipywidgets.widgets.Image(value=videos[i], width=150, height=150)
    true_class = LABELS[ground_truths[i]]
    pred_class = LABELS[preds[i]]
    caption = f"T: {true_class} | P: {pred_class}"

    boxes.append(make_box_for_grid(ib, caption))

ipywidgets.widgets.GridBox(
    boxes, layout=ipywidgets.widgets.Layout(grid_template_columns="repeat(5, 200px)")
)